In [6]:
import duckdb
import pandas as pd
import os
os.chdir(r'C:\Users\Tobi\Desktop\CSHW\TransitKPIFramework')
con = duckdb.connect('data/transit_kpi.duckdb')

# Verify tables available
print(con.sql("SHOW TABLES").df().to_string())

             name
0        calendar
1  calendar_dates
2          gtfsrt
3      stop_times
4     trip_delays
5           trips


In [7]:
con.sql("""
    SELECT service_id FROM calendar
    WHERE tuesday = 1
    AND start_date <= 20260407
    AND end_date >= 20260407
    
    UNION
    
    SELECT service_id FROM calendar_dates
    WHERE date = 20260407
    AND exception_type = 1
""").df()

,service_id
0,16
1,28
2,308746
3,46
4,33610
5,22
6,52
7,341252
8,40
9,390240


In [8]:
con.execute("DROP TABLE IF EXISTS trip_delays")

con.execute("""
    CREATE TABLE trip_delays AS
    WITH active_services AS (
        -- Regular weekday services
        SELECT service_id FROM calendar
        WHERE tuesday = 1
        AND start_date <= 20260407
        AND end_date >= 20260407

        UNION

        -- Exception services added on April 7th
        SELECT service_id FROM calendar_dates
        WHERE date = 20260407
        AND exception_type = 1
    ),
    matched AS (
        -- Join GTFS-RT to static schedule via trip_id + stop_sequence
        -- filtered to valid service IDs for this date
        SELECT
            rt.trip_id,
            rt.route_id,
            rt.service_date,
            rt.snapshot_ts,
            rt.stop_sequence,
            rt.stop_id,
            rt.schedule_relationship,
            rt.predicted_unix,
            rt.predicted_ts,
            rt.midnight_unix,
            st.arrival_time                 AS scheduled_arrival_time,
            st.arrival_seconds              AS scheduled_arrival_seconds
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN active_services a
            ON t.service_id = a.service_id
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
            AND rt.stop_sequence = st.stop_sequence
        WHERE rt.route_id IN ('23', '47')
        AND rt.schedule_relationship = 0
    ),
    nearest_day AS (
        -- For each record pick the scheduled unix timestamp
        -- that is closest to the predicted unix timestamp
        -- handles overnight trips and day boundary crossings cleanly
        SELECT
            *,
            CASE
                WHEN ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds - 86400))
                   < ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds))
                 AND ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds - 86400))
                   < ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds + 86400))
                THEN midnight_unix + scheduled_arrival_seconds - 86400
                WHEN ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds + 86400))
                   < ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds))
                THEN midnight_unix + scheduled_arrival_seconds + 86400
                ELSE midnight_unix + scheduled_arrival_seconds
            END AS scheduled_unix
        FROM matched
    ),
    deduped AS (
        -- Keep only the snapshot closest to scheduled arrival
        -- one record per unique trip + stop event
        SELECT
            *,
            ROUND((predicted_unix - scheduled_unix) / 60.0, 1) AS delay_minutes,
            ROW_NUMBER() OVER (
                PARTITION BY trip_id, stop_sequence
                ORDER BY ABS(predicted_unix - scheduled_unix)
            ) AS snapshot_rank
        FROM nearest_day
    )
    SELECT
        trip_id,
        route_id,
        service_date,
        snapshot_ts,
        stop_sequence,
        stop_id,
        schedule_relationship,
        scheduled_arrival_time,
        scheduled_arrival_seconds,
        scheduled_unix,
        predicted_unix,
        predicted_ts,
        delay_minutes
    FROM deduped
    WHERE snapshot_rank = 1
""")

print(con.sql("SELECT COUNT(*) as rows FROM trip_delays").df().to_string())

    rows
0  42041


In [9]:
print(con.sql("""
    SELECT
        CASE
            WHEN delay_minutes < -60  THEN 'bad match (< -60 min)'
            WHEN delay_minutes < -30  THEN 'very early (-60 to -30 min)'
            WHEN delay_minutes < -1   THEN 'early (-30 to -1 min)'
            WHEN delay_minutes <= 5   THEN 'on time (-1 to +5 min)'
            WHEN delay_minutes <= 15  THEN 'late (5 to 15 min)'
            WHEN delay_minutes <= 60  THEN 'very late (15 to 60 min)'
            ELSE                           'bad match (> 60 min)'
        END          AS bucket,
        COUNT(*)     AS records,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM trip_delays
    GROUP BY bucket
    ORDER BY MIN(delay_minutes)
""").df().to_string())

                     bucket  records   pct
0     early (-30 to -1 min)     4396  10.5
1    on time (-1 to +5 min)    34880  83.0
2        late (5 to 15 min)     2231   5.3
3  very late (15 to 60 min)      534   1.3


In [10]:
print(con.sql("""
    SELECT
        route_id,
        COUNT(*)                                                          AS total_stop_events,
        ROUND(
            SUM(CASE WHEN delay_minutes BETWEEN 0 AND 3 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
        , 1)                                                              AS otp_0_3,
        ROUND(
            SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
        , 1)                                                              AS otp_1_5,
        ROUND(
            SUM(CASE WHEN delay_minutes BETWEEN -2 AND 7 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
        , 1)                                                              AS otp_2_7
    FROM trip_delays
    GROUP BY route_id
    ORDER BY route_id
""").df().to_string())

  route_id  total_stop_events  otp_0_3  otp_1_5  otp_2_7
0       23              21706     49.2     85.9     94.5
1       47              20335     44.2     79.8     91.3
